# Práctica: Reconocimiento de Números Escritos a Mano
**Fundamentos de Inteligencia Artificial**

---
- **Nombre:** ______________________________
- **Grupo:** ________
- **Fecha:** 23 de noviembre de 2026
- **Modalidad:** Individual, en Google Colab
- **Algoritmos:** KNN, SVM y Red Neuronal
---


## Paso 1 · Importar las herramientas
Cargamos `numpy` y `pandas` para manejar números y tablas, `matplotlib` para dibujar y `scikit-learn` para los modelos. Colab ya trae todas estas bibliotecas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
print("Herramientas importadas correctamente.")

## Paso 2 · Cargar los datos
El conjunto de números viene incluido en scikit-learn, así que no hay que descargar nada. Guardamos en `X` los números de cada imagen y en `y` el dígito correcto.

In [ ]:
digitos = load_digits()
X = digitos.data # las imágenes convertidas en números
y = digitos.target # el número que representa cada imagen (0 a 9)
print("Imágenes:", X.shape[0])
print("Números por imagen:", X.shape[1])

## Paso 3 · Ver las imágenes
Dibujamos las primeras 20 imágenes con su etiqueta encima y contamos cuántas imágenes hay de cada número.

In [ ]:
fig, ejes = plt.subplots(2, 10, figsize=(12, 3))
for i, eje in enumerate(ejes.ravel()):
    eje.imshow(digitos.images[i], cmap="gray_r")
    eje.set_title(y[i])
    eje.axis("off")
plt.show()

print("Imágenes de cada número:")
print(pd.Series(y).value_counts().sort_index())

### ✍️ Pregunta 1
**¿Cuántas imágenes hay de cada número? ¿Están equilibradas? ¿Qué números crees que serán más difíciles de distinguir entre sí?**

**Respuesta:**
- **Cantidad por número:**
  - 0: 178 | 1: 182 | 2: 177 | 3: 183 | 4: 181
  - 5: 182 | 6: 181 | 7: 179 | 8: 174 | 9: 180
  - **Total de imágenes:** 1,797.
- **¿Están equilibradas?:** Sí, el conjunto de datos está notablemente balanceado. Cada dígito tiene entre 174 y 183 muestras, lo que significa que cada clase representa aproximadamente un 10% del total. No existe ningún desbalance ni sesgo hacia algún dígito en particular.
- **Números más difíciles de distinguir:**
  - **1 y 7:** Ambos comparten un eje vertical predominante en el centro; en una cuadrícula tan pequeña de 8×8 píxeles, la barra superior horizontal del 7 puede ser muy tenue o ausente.
  - **3 y 8:** Ambos tienen dos curvas en la mitad derecha; si la parte izquierda del 8 no está completamente cerrada o se escribe con trazo fino, la distribución de intensidades es casi idéntica a la de un 3.
  - **9 y 4 (o 9 y 8):** El bucle superior cerrado del 9 suele asemejarse al bucle superior del 8, o a un 4 cuya parte superior queda cerrada.

## Paso 4 · Una imagen es una tabla de números
Imprimimos la primera imagen como tabla de 8 × 8 y la dibujamos con el valor de cada píxel escrito encima (0 = blanco, 16 = negro).

In [ ]:
print("El número de la imagen es:", y[0])
print(digitos.images[0].astype(int))

plt.figure(figsize=(4, 4))
plt.imshow(digitos.images[0], cmap="gray_r")
for fila in range(8):
    for col in range(8):
        valor = int(digitos.images[0][fila, col])
        plt.text(col, fila, valor, ha="center", va="center",
                 color="white" if valor > 8 else "black")
plt.axis("off")
plt.show()

## Paso 5 · Separar entrenamiento y prueba
Guardamos 25 % de las imágenes para el examen final. `stratify=y` mantiene la misma proporción de cada número en ambos conjuntos y `random_state=42` hace que todos obtengan la misma división.

In [ ]:
X_ent, X_prueba, y_ent, y_prueba = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print("Entrenamiento:", len(X_ent))
print("Prueba:", len(X_prueba))

## Paso 6 · La línea base: un modelo que no aprende
`DummyClassifier` siempre responde el número más común, sin mirar la imagen. Sirve como punto de comparación: cualquier modelo útil debe superarlo por mucho.

In [ ]:
tonto = DummyClassifier(strategy="most_frequent").fit(X_ent, y_ent)
print("Exactitud de la línea base:", round(tonto.score(X_prueba, y_prueba), 3))

## Paso 7 · Entrenar tres modelos
- **KNN:** busca las 3 imágenes más parecidas del entrenamiento y vota.
- **SVM:** traza fronteras entre los 10 números, cada una con el margen más ancho posible.
- **Red neuronal:** 64 entradas (una por píxel), 64 neuronas ocultas y 10 salidas, una por número.

`fit` es el momento en que cada modelo aprende de las 1,347 imágenes de entrenamiento.

In [ ]:
modelos = {
    "KNN (k=3)": KNeighborsClassifier(n_neighbors=3),
    "SVM": SVC(),
    "Red neuronal": MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42),
}
for nombre, modelo in modelos.items():
    modelo.fit(X_ent, y_ent)
    print("Entrenado:", nombre)

## Paso 8 · Evaluar con el conjunto de prueba
Presentamos el examen final: cada modelo clasifica las 450 imágenes de prueba. Comparamos la exactitud de entrenamiento con la de prueba y contamos los errores.

In [ ]:
filas = []
for nombre, modelo in modelos.items():
    p = modelo.predict(X_prueba)
    filas.append({
        "Modelo": nombre,
        "Exactitud entrenamiento": modelo.score(X_ent, y_ent),
        "Exactitud prueba": accuracy_score(y_prueba, p),
        "Errores": int((p != y_prueba).sum())
    })
print(pd.DataFrame(filas).round(3))

ConfusionMatrixDisplay.from_estimator(modelos["SVM"], X_prueba, y_prueba, cmap="Blues")
plt.title("SVM: filas = número real, columnas = predicción")
plt.show()

### ✍️ Pregunta 2
**¿Qué modelo tuvo menos errores? ¿Alguno tiene una exactitud de entrenamiento mucho mayor que la de prueba? ¿Qué significa eso?**

**Respuesta:**
- **Modelo con menos errores:** **SVM** obtuvo el mejor desempeño general con solo **4 errores** en las 450 imágenes de prueba, logrando una exactitud de prueba del **99.1% (0.991)** frente al 98.4% de KNN (7 errores) y al 96.2% de la Red Neuronal (17 errores).
- **Diferencia entre entrenamiento y prueba:** La **Red Neuronal (MLP)** obtuvo una exactitud del **100% (1.000)** en entrenamiento, pero cayó a **96.2% (0.962)** en prueba, presentando una diferencia de casi 4 puntos porcentuales y 17 fallos.
- **Significado (Sobreajuste / Overfitting):** Esto significa que la Red Neuronal sufrió de **sobreajuste**. Al tener una gran cantidad de pesos libres, memorizó patrones de ruido y particularidades caligráficas exclusivas del conjunto de entrenamiento en lugar de abstraer las formas matemáticas esenciales, lo que redujo su capacidad de generalización ante imágenes nuevas.

### ✍️ Pregunta 3
**En la matriz de confusión, ¿qué números se confunden? ¿Por qué crees que pasa?**

**Respuesta:**
En la matriz de confusión del modelo SVM, que solo cometió 4 equivocaciones en total, se identifican las siguientes confusiones:
1. **El número 8 confundido con el 1 (2 casos):** Los dígitos reales 8 fueron clasificados como 1. Esto pasa cuando la persona escribe un 8 muy comprimido horizontalmente o con trazos laterales muy suaves, lo que en una matriz de 8×8 píxeles deja solo una franja vertical central visible, idéntica a la estructura de un 1.
2. **El número 9 confundido con el 7 (1 caso):** Ocurre cuando el lazo superior del 9 no se cierra completamente o queda muy plano; el modelo detecta la línea horizontal superior y la diagonal inferior, prediciendo un 7.
3. **El número 5 confundido con el 9 (1 caso):** Si la curva inferior del 5 se une o aproxima demasiado al trazo medio superior, el algoritmo lo interpreta como un bucle cerrado superior con mástil.
**Causa de fondo:** En una resolución minúscula de 8×8 (64 píxeles en escala de grises de 0 a 16), se pierde la micro-estructura de las curvas y los pequeños espacios vacíos, haciendo que trazos manuscritos atípicos proyecten intensidades casi indistinguibles.

## Paso 9 · Ver los errores
Dibujamos las imágenes de prueba en las que el SVM se equivocó, con el número real y lo que respondió.

In [ ]:
p = modelos["SVM"].predict(X_prueba)
errores = np.where(p != y_prueba)[0]
fig, ejes = plt.subplots(1, len(errores), figsize=(2.2 * len(errores), 2.5))
for eje, i in zip(np.atleast_1d(ejes), errores):
    eje.imshow(X_prueba[i].reshape(8, 8), cmap="gray_r")
    eje.set_title(f"Real: {y_prueba[i]}\nDijo: {p[i]}")
    eje.axis("off")
plt.show()

## Paso 10 · Dibuja tu propio número
Escribimos dos tablas de 8 × 8 con valores del 0 (blanco) al 16 (negro):
1. **Número Reconocible:** Un 7 claro y nítido.
2. **Número Confuso:** Un dígito ambiguo con trazo vertical de 1 y cabezal superior ensanchado tipo 3/7 para provocar discrepancia.

In [ ]:
# 1. Número claramente reconocible (7)
numero_reconocible = np.array([
    [0, 8, 16, 16, 16, 16,  8, 0],
    [0, 4,  8,  8,  8, 16, 10, 0],
    [0, 0,  0,  0,  6, 16,  4, 0],
    [0, 0,  0,  0, 14, 10,  0, 0],
    [0, 0,  0,  6, 16,  2,  0, 0],
    [0, 0,  0, 14, 10,  0,  0, 0],
    [0, 0,  4, 16,  4,  0,  0, 0],
    [0, 0,  8, 12,  0,  0,  0, 0],
])

# 2. Número confuso (híbrido ambiguo 1 / 3 / 7)
numero_confuso = np.array([
    [0, 0, 10, 15, 14,  0,  0, 0],
    [0, 0,  0,  2, 14,  0,  0, 0],
    [0, 0,  0,  2, 14,  0,  0, 0],
    [0, 0,  0,  4, 14,  0,  0, 0],
    [0, 0,  0,  4, 14,  0,  0, 0],
    [0, 0,  0,  2, 14,  0,  0, 0],
    [0, 0,  0,  2, 14,  0,  0, 0],
    [0, 0,  0,  2, 14,  0,  0, 0],
])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5, 2.5))
ax1.imshow(numero_reconocible, cmap="gray_r")
ax1.set_title("Número Reconocible (7)")
ax1.axis("off")
ax2.imshow(numero_confuso, cmap="gray_r")
ax2.set_title("Número Confuso (1 vs 3)")
ax2.axis("off")
plt.show()

print("--- Predicciones para Dígito Reconocible ---")
for nombre, modelo in modelos.items():
    print(nombre, "dice que es un:", modelo.predict([numero_reconocible.ravel()])[0])

print("\n--- Predicciones para Dígito Confuso ---")
for nombre, modelo in modelos.items():
    print(nombre, "dice que es un:", modelo.predict([numero_confuso.ravel()])[0])

### ✍️ Pregunta 4
**Dibuja dos números propios: uno que los modelos reconozcan y otro que los confunda. ¿Qué cambiaste para confundirlos?**

**Respuesta:**
1. **Número que reconocen con facilidad (7):** Se construyó respetando la morfología canónica: una línea superior continua y un trazo diagonal central limpio. Todos los modelos (KNN, SVM y Red Neuronal) respondieron unánimemente que es un **7**.
2. **Número que los confunde:** Diseñamos una columna vertical uniforme correspondiente a un 1, pero le añadimos un lóbulo o cabezal horizontal grueso a la izquierda en la fila 0 (valores 10, 15, 14). Los resultados fueron:
   - **KNN dijo:** **1** (por cercanía Euclidiana a las columnas verticales de entrenamiento).
   - **SVM dijo:** **1** (al ubicar el vector de soporte más cercano a la frontera del 1).
   - **Red Neuronal dijo:** **3** (las neuronas de capas intermedias se activaron por el bloque superior izquierdo confundiéndolo con la curva superior de un 3).
3. **¿Qué se cambió para confundirlos?:** Modificamos la coherencia geométrica y las intensidades locales. Al colocar píxeles de alta densidad en la esquina superior izquierda de un tallo vertical, rompimos la simetría del 1 y creamos un rasgo no lineal que activa detectores de arcos de otros dígitos, logrando que los algoritmos discrepen entre sí.

## 5. Conclusiones
**En cinco líneas, explica qué modelo usarías para leer códigos postales y por qué. Usa al menos dos números de tus resultados.**

1. Para clasificar códigos postales seleccionaría el modelo **SVM** debido a su confiabilidad y mínima tasa de fallos.
2. En el conjunto de prueba alcanzó una exactitud del **99.1%**, cometiendo únicamente **4 errores** de 450 imágenes.
3. Superó a la Red Neuronal, la cual sufrió sobreajuste severo cayendo al 96.2% de exactitud con 17 equivocaciones.
4. A diferencia de KNN, una vez entrenado, SVM no requiere comparar cada imagen contra las 1,347 muestras de entrenamiento en disco.
5. En conclusión, SVM garantiza un procesamiento postal veloz en tiempo real y reduce drásticamente las cartas desviadas.